# 03 Karnataka market prices: unit audit and availability

Question (T108 only): what is the unit of the Karnataka market prices, how do prices vary by
market and season, how often do they spike, and how many future labels exist? **Today none of
this can be measured: both price sources are blocked and there are zero price rows.**
S22 (India Data Portal Agmarknet) has no reachable download, licence or units (T023). S23
(data.gov.in mandi prices) has a verified open licence (GODL) but no publisher statement of the
price denominator (per quintal, per kg), so its numbers are unlabelled (T025). Both adapters are
disabled and raise `SourceUnavailable` before any I/O; a disabled source never returns synthetic
rows (M06 is blocked for the same reason).

So this notebook is an **audit**. The unit audit lists every field of each source's frozen
discovery record with its unit status. Market-season patterns, spike prevalence and future-label
coverage are computed by tested helpers over the zero rows: they report n = 0 and "not
computable", never 0% and never a filled label. No feature is built, so none can use future data.

Run from the repository root in a fresh Python 3.12 process. Outputs go to
`reports/eda/03_ka_prices/` (or `ATLAS_EDA_OUT`).

In [ ]:
import csv
import json
import os
import textwrap
from collections import Counter
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

from atlas.analysis.prices import (
    audit_rows,
    label_summary,
    load_record,
    month_counts,
    source_status,
)
from atlas.analysis.quality import caption
from atlas.config import load_state
from atlas.contracts import RawArtifact
from atlas.ingest.base import SourceUnavailable
from atlas.ingest.s22 import S22
from atlas.ingest.s23 import S23

OUT = Path(os.environ.get("ATLAS_EDA_OUT", "reports/eda/03_ka_prices"))
OUT.mkdir(parents=True, exist_ok=True)
SOURCES = {"S22": S22, "S23": S23}
ka = load_state(Path("configs/states/ka.yaml"))
records = {s: load_record(Path(f"configs/sources/{s.lower()}.yaml")) for s in SOURCES}


def write_csv(name, rows):
    with open(OUT / name, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]), lineterminator="\n")
        w.writeheader()
        w.writerows(rows)

In [ ]:
audit = [r for rec in records.values() for r in audit_rows(rec)]
status = [source_status(rec) for rec in records.values()]
write_csv("unit_audit.csv", audit)
write_csv("source_status.csv", status)
by_status = {s: Counter(r["unit_status"] for r in audit if r["source"] == s) for s in SOURCES}
for s in status:
    print(s["source"], s["outcome"], "licence", s["licence"], "units", s["units"])
print(len(audit), "fields;", {s: dict(c) for s, c in by_status.items()})

In [ ]:
# The adapters refuse before any I/O; a frozen evidence record is only the argument they get.
blocked = {}
for sid, adapter in SOURCES.items():
    manifest = json.loads(Path(f"tests/fixtures/{sid.lower()}/manifest.json").read_text("utf-8"))
    e = manifest["evidence"][0]
    raw = RawArtifact(
        sha256=e["sha256"],
        path=Path("data/raw") / e["raw_path"],
        fetched_at=e["fetched_at"],
        media_type=e["media_type"],
        response_status=e["status"],
        source_url=e["url"],
        schema_mapping_version=f"{sid.lower()}-v1",
    )
    try:
        rows = adapter().parse(raw, ka.source(sid))
    except SourceUnavailable as exc:
        blocked[sid] = str(exc)
    else:
        raise AssertionError(f"{sid} returned {rows.num_rows} rows; it must be blocked")
prices = []  # price rows: none exist
reason = "source blocked, no price rows (" + ", ".join(sorted(blocked)) + ")"
season = month_counts(p["date"] for p in prices)
measures = label_summary((p["spike_label"] for p in prices), reason)
write_csv("measures.csv", measures)
print(season["n"], "price rows;", [m["status"] for m in measures])

In [ ]:
cap = caption(
    "S22 and S23 discovery records (price sources, blocked)",
    "price fields by unit status",
    "fields",
    min(r["discovered_at"] for r in status)[:10],
    max(r["discovered_at"] for r in status)[:10],
    f"{len(audit)} fields; 0 price rows",
)
labels = sorted({k for c in by_status.values() for k in c})
fig, ax = plt.subplots(figsize=(6, 4))
fig.subplots_adjust(bottom=0.2)
for i, sid in enumerate(SOURCES):
    xs = [j + 0.4 * i for j in range(len(labels))]
    ax.bar(xs, [by_status[sid].get(k, 0) for k in labels], width=0.4, label=sid)
ax.set_xticks([j + 0.2 for j in range(len(labels))], labels)
ax.set_ylabel("Fields in discovery record")
ax.set_title("Price fields by unit status (no price rows exist)", fontsize=10)
ax.legend()
fig.text(0.01, 0.005, textwrap.fill(cap, 100), fontsize=5)
fig.savefig(OUT / "field_unit_status.png", dpi=80, metadata={"Software": None})
plt.close(fig)

In [ ]:
summary = {
    "price_rows": len(prices),
    "sources": {
        s["source"]: s
        | {
            "fields": sum(by_status[s["source"]].values()),
            "fields_by_unit_status": dict(sorted(by_status[s["source"]].items())),
            "adapter_error": blocked[s["source"]],
        }
        for s in status
    },
    "market_season": season | {"status": f"not computable: {reason}"},
    "measures": measures,
    "charts": ["field_unit_status.png"],
    "caption": cap,
    "limits": [
        "Zero price rows: S22 and S23 are blocked, so no market, season or spike is measured.",
        "S23 units are unlabelled: price per quintal vs per kg stays unverified (T025).",
        "S23 history is one current-day snapshot (5 Nov 2026), not the 2021-2026 history planned.",
        "Measures with n = 0 are not computable; no 0% share and no label is filled.",
        "No feature is built, so none can use data after the frozen discovery dates.",
    ],
}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2) + "\n", "utf-8", newline="\n")
print(json.dumps(summary, indent=2))